## Question 4 : Vertibi Algorithm

# Hidden Markov Model (HMM)

A **Hidden Markov Model (HMM)** is a statistical model used to represent systems that follow a Markov process with hidden states. HMMs are particularly useful in situations where the system is not directly observable, but we can infer information from observable data. 

## Key Components of HMM

1. **States**:
   - The system is assumed to be in one of a finite number of states at any given time.
   - These states are "hidden" because we cannot observe them directly.
   - Example: In a biological sequence model, states could represent biological features like "exon" and "intron."

2. **Observations**:
   - Each state produces an observable symbol according to some probability distribution.
   - These observations are what we can see or measure.
   - Example: In a DNA sequence model, the observations could be the nucleotides ('A', 'C', 'G', 'T').

3. **Transition Probabilities**:
   - The probability of transitioning from one state to another.
   - These probabilities form a matrix called the **transition matrix**.
   - Example: The probability of moving from state "exon" to state "intron" could be 0.1.

4. **Emission Probabilities**:
   - The probability of observing a particular symbol from a given state.
   - These probabilities are captured in an **emission matrix**.
   - Example: The probability of observing nucleotide 'A' when in the "exon" state could be 0.25.

5. **Initial Probabilities**:
   - The probability of the system starting in each state.
   - This forms a distribution over the initial states.

## Model Assumptions

- **Markov Property**: The future state of the system depends only on the current state and not on the previous states (memoryless property).
- **Stationary Process**: The system behaves the same way over time, meaning the transition probabilities do not change.

## Applications

HMMs are used in various domains, such as:

- **Speech recognition**: Modeling the sequence of spoken words and their corresponding acoustic features.
- **Bioinformatics**: Modeling biological sequences, such as DNA or protein sequences.
- **Finance**: Modeling stock price movements.
- **Natural Language Processing (NLP)**: Part-of-speech tagging, named entity recognition, and other sequence labeling tasks.

## Key Algorithms

1. **Forward Algorithm**: Computes the probability of a given sequence of observations.
2. **Viterbi Algorithm**: Finds the most likely sequence of hidden states for a given sequence of observations.
3. **Baum-Welch Algorithm**: An Expectation-Maximization (EM) algorithm used for training the model (i.e., estimating the transition and emission probabilities).

## Summary

HMMs are powerful tools for modeling sequential data, where the sequence of observations is influenced by underlying hidden states. By leveraging probabilistic methods, HMMs can be used to make predictions, classify sequences, and perform other tasks based on observed data.


In [9]:
import numpy as np

states = ['E', '5', 'I']
nucleotides = ['A', 'C', 'G', 'T']

# 2. Initial Probabilities: 
# Considering equal probabilities for both the states in the starting
initial_probabilities = {'E': 1.0, '5': 0.0, 'I': 0.0}

# 3. Transition matrix: Probabilities of moving from one state to another
transition_probabilities = {
    'Start': {'E':1.0, '5': 0, 'I': 0.0, 'End':0.0},
    'E': {'E': 0.9, '5': 0.1 , 'I': 0.0, 'End': 0.0},
    '5': {'E': 0.0, '5': 0.0 , 'I': 1.0, 'End': 0.0},
    'I': {'E': 0.0, '5': 0.0 , 'I': 0.9, 'End': 0.1}
}

# 4. Emission probabilities: Probability of emitting nucleotides (A, C, G, T) in each state
emission_probs = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.00, 'G': 0.95, 'T': 0.00},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4},
}

In [2]:
import math
def log(x):
    if(x == 0):
        return -math.inf
    else:
        return math.log(x)

In [ ]:
# Log probability of given path: This function will be used to compute the log probability of a state path emitting an observed sequence. (base 2)

def get_log_prob_of_a_given_path(state_path, observed_sequence):
    log_prob = 0.0
    if len(state_path)!=len(observed_sequence):
        raise ValueError("The length of state path and the observed sequence must be same")
    prev_state = 'Start'

    for i in range(len(observed_sequence)):
        current_state = state_path[i]
        observed_state = observed_sequence[i]
        log_prob += log(transition_probabilities[prev_state][current_state])+log(emission_probs[current_state][observed_state])
        prev_state = current_state
    if(prev_state == 'I'):
        log_prob += log(transition_probabilities[prev_state]['End'])

    return log_prob

In [4]:
state_path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
observed_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"
ans = get_log_prob_of_a_given_path(state_path, observed_sequence)

print(ans)

-41.21967768602254


## The Viterbi Algorithm

1. **INITIALISATION**: Set up the first column of Viterbi matrix with initial probabilities
2. **RECURSION**: For each position in the sequence, calculate:
   - For each possible state, find the most likely previous state
   - Update our matrix with the new probability
   - Keep track of the dp (backpointers) to remember which path we took
3. **TERMINATE**: Find the most likely final state
4. **BACKTRACK**: Follow these backpointers to reconstruct the most likely state sequence


One interesting thing is that depending on our models parameters, the most likely path might actually just be all exons when transition probabilities heavily favor staying in one state.
Also I am amazed to see this algorithm is used all over the place like speech recognition , spell check , etc.


In [ ]:
# Viterbi Algorithm Implementation

def viterbiAlgo(observed_sequence):
    num_states = len(states)
    num_observations = len(observed_sequence)
    viterbi_matrix = np.full((num_states, num_observations), -np.inf)
    # backpointers will used for backtracking (retracing the path of maxima)
    backpointers = np.zeros((num_states, num_observations), dtype=int)
    # Convert to state indices
    state_to_index = {s: i for i, s in enumerate(states)}

    # Initializing the DP table
    for i, j in enumerate(states):
        viterbi_matrix[i, 0] = log(initial_probabilities[j]) + \
        log(emission_probs[j][observed_sequence[0]])
        # Recursive part
    for k in range(1, num_observations):
        for curr_index, current_state in enumerate(states):
            max_log_prob = -np.inf
            best_prev_idx = 0
            for prev_idx, prev_state in enumerate(states):
                log_prob = viterbi_matrix[prev_idx, k-1] + \
                log(transition_probabilities[prev_state][current_state])
                if log_prob > max_log_prob:
                    max_log_prob = log_prob
                    best_prev_idx = prev_idx
            viterbi_matrix[curr_index, k] = max_log_prob + \
            log(emission_probs[current_state][observed_sequence[k]])
            backpointers[curr_index, k] = best_prev_idx

    bestPath = []
    bestFinalIndex = np.argmax(viterbi_matrix[:, -1])
    bestPath.append(states[bestFinalIndex])

    for i in range(num_observations-1, 0, -1):
        bestFinalIndex = backpointers[bestFinalIndex, i]
        bestPath.insert(0, states[bestFinalIndex])
    return bestPath, np.max(viterbi_matrix[:, -1])

In [7]:
observed_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"
best_path, log_prob = viterbiAlgo(observed_sequence)
print(f"Most probable path is {best_path} " )
print(f"Log probability for path {log_prob}" )

Most probable path is ['E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E'] 
Log probability for path -38.677666280562796
